In [ ]:
#------------------------------------------------ Import Lib ----------------------------------------
import re
import os
import datetime
from time import sleep

import requests
from bs4 import BeautifulSoup
import pandas as pd

import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


In [ ]:
#------------------------------------------------ Begin_ fileName ----------------------------------------

regulatorName = 'LK CBSL' ## change to current controller name

print(f"Running {regulatorName} Web Scraping Tool v.1.0")

now = datetime.datetime.now()

filename = '{} SQL Ready {}.xlsx'.format(regulatorName, str(now).replace(":", ".")[:-7])

try:
    scriptfolder = os.path.dirname(os.path.abspath(__file__)) ## production environment (.py)
except NameError:
    scriptfolder = os.getcwd() ## notebook environment

os.chdir(scriptfolder)

tempfolder = os.path.join(scriptfolder, 'tempfolder') # output .xlsx is saved here

if os.path.exists(tempfolder):
    for rem in os.listdir(tempfolder):
        os.remove(os.path.join(tempfolder, rem))
else:
    os.mkdir(tempfolder)


In [ ]:
#------------------------------------------------ Begin_Variable ----------------------------------------
sqldict={'bvdid': [], 'priority': [], 'ListLabel': [], 'Typology': [], 'EntryType': [], 'Name': [], 'InternalID_1': [], 'InternalID_1_type': [], 'InternalID_2': [],
          'InternalID_2_type': [], 'InternalID_3': [], 'InternalID_3_type': [], 'CoType': [], 'License_Type': [], 'Address_1': [], 'Address_2': [], 'City': [],
          'Zip': [], 'Cntry': [], 'Phone': [], 'Fax': [], 'Website': [], 'Email': [], 'RegulationType': [], 'RegulationTypeCode': [], 'RegulationDate': [], 'CancellationDate': [],
          'RegCtry': [], 'RegCode' : [], 'ListCode': [], 'ListLanguage': [], 'ListValidityDate': [], 'ListName': [], 'ListProcessDate': [], 'LEI Code': [], 'BIC SWIFT Code': [], 'Name - Mother Company': [],
          'Address_1 - Mother company': [], 'Address_2 -  Mother company': [], 'City - Mother company': [], 'Zip - Mother company': [], 'Cntry - Mother company': [],
          'Phone - Mother company': []}


regdict={

        regulatorName+' 1': 'https://www.cbsl.gov.lk/authorized-financial-institutions/licensed-commercial-banks',
        regulatorName+' 2': 'https://www.cbsl.gov.lk/authorized-financial-institutions/licensed-finance-companies',
        regulatorName+' 3': 'https://www.cbsl.gov.lk/authorized-financial-institutions/registered-finance-leasing-establishments',
        regulatorName+' 4': 'https://www.cbsl.gov.lk/authorized-financial-institutions/licensed-specialised-banks',
        regulatorName+' 5': 'https://www.cbsl.gov.lk/authorized-financial-institutions/registered-authorised-primary-dealers',
        regulatorName+' 6': 'https://www.cbsl.gov.lk/authorized-financial-institutions/authorized-money-broking-companies',
        regulatorName+' 7': 'https://www.cbsl.gov.lk/authorized-financial-institutions/licensed-microfinance-companies',
        }


Typology={

       regulatorName + ' 1': 'Licensed Commercial Banks',
       regulatorName + ' 2': 'Licensed Finance Companies',
       regulatorName + ' 3': 'Registered Finance Leasing Establishments',
       regulatorName + ' 4': 'Licensed Specialised Banks',
       regulatorName + ' 5': 'Authorised Primary Dealers',
       regulatorName + ' 6': 'Authorized Money Broking Companies',
       regulatorName + ' 7': 'Licensed Microfinance Institutions',

        }


# ListLabel rule (per ticket owner): 1 = bank named in list name, 2 = insurance,
# 3 = bank & insurance, 4 = other
ListLabeldict={

       regulatorName + ' 1': 1,   # Licensed Commercial Banks
       regulatorName + ' 2': 4,   # Licensed Finance Companies
       regulatorName + ' 3': 4,   # Registered Finance Leasing Establishments
       regulatorName + ' 4': 1,   # Licensed Specialised Banks
       regulatorName + ' 5': 4,   # Authorised Primary Dealers
       regulatorName + ' 6': 4,   # Authorized Money Broking Companies
       regulatorName + ' 7': 4,   # Licensed Microfinance Institutions

        }


processdate = now.strftime('%Y-%m-%d')

headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/126.0.0.0 Safari/537.36'}

In [ ]:
#------------------------------------------------ Begin_Fouction ----------------------------------------

def bourange_same_length_array(sqldict) :

    maxlen = len(sqldict['ListProcessDate'])

    for key, val in sqldict.items():

        if len(sqldict[key]) != maxlen:

            empty = []

            total_empty = maxlen - len(sqldict[key])

            for i in range(total_empty):

                empty.append('')

            sqldict[key]=sqldict[key]+empty

    return sqldict


# bold segments in the Name cell that are footnotes, not part of the entity name
NOTE_KEYWORDS = ('suspend', 'refer note', 'restrain', 'w.e.f')


def is_note_text(txt):
    t = txt.lower()
    return txt.strip() == '*' or any(k in t for k in NOTE_KEYWORDS)


def cell_lines(cell):
    """Split a <td> into visual lines (on <br> and <p> boundaries), whitespace-normalised."""
    for br in cell.find_all('br'):
        br.replace_with('\n')
    for p in cell.find_all('p'):
        p.insert_after('\n')
    lines = []
    for ln in cell.get_text().split('\n'):
        ln = re.sub(r'\s+', ' ', ln).strip()
        if ln:
            lines.append(ln)
    return lines


def extract_name_and_address(cell):
    """Name = bold (strong/b) segments joined (footnote bolds excluded); address = remaining lines."""
    strongs = [s for s in cell.find_all(['strong', 'b']) if s.find_parent(['strong', 'b']) is None]
    parts = []
    for s in strongs:
        t = re.sub(r'\s+', ' ', s.get_text(' ', strip=True)).strip()
        if not t or is_note_text(t):
            continue
        parts.append(t)
    name = re.sub(r'\s+', ' ', ' '.join(parts)).strip().rstrip('*').strip()
    for s in strongs:
        s.decompose()
    # drop parenthetical leftovers such as "(Formerly Ceylinco Shriram Securities Ltd)"
    address_lines = [ln for ln in cell_lines(cell) if not ln.startswith('(') or re.match(r'^\(\d', ln)]
    # repair a closing bracket left outside the bold name, e.g. "...(Primary Dealer Unit" + "), No.21, ..."
    if name.count('(') > name.count(')') and address_lines and address_lines[0].startswith(')'):
        name = re.sub(r'\s+\)', ')', name + ')')
        address_lines[0] = address_lines[0].lstrip(') ,').strip()
        if not address_lines[0]:
            address_lines = address_lines[1:]
    address = ', '.join(ln.rstrip(',').strip() for ln in address_lines)
    return name, address


def extract_contacts(label_cell, value_cell):
    """Return (phone, fax, email, website) from the two 'Contact Details' columns."""
    labels = cell_lines(label_cell)
    values = cell_lines(value_cell)
    phone = fax = email = website = ''
    if len(labels) == len(values):
        # line-aligned Tel./Fax/E-mail/Website labels
        for lab, val in zip(labels, values):
            v = val.strip()
            if v in ('-', '--', ''):
                continue
            l = lab.lower()
            if 'tel' in l and not phone:
                phone = v
            elif 'fax' in l and not fax:
                fax = v
            elif 'mail' in l and not email:
                email = v
            elif 'web' in l and not website:
                website = v
    else:
        # labels merged on one line (e.g. money brokers) -> classify values by pattern
        numbers = []
        for v in values:
            v = v.strip()
            if v in ('-', '--', ''):
                continue
            low = v.lower()
            if '@' in v:
                if not email:
                    email = v
            elif low.startswith('www') or low.startswith('http'):
                if not website:
                    website = v
            elif len(re.sub(r'\D', '', v)) >= 5:
                numbers.append(v)
        if numbers:
            phone = numbers[0]
        if len(numbers) > 1:
            fax = numbers[1]
    website = re.sub(r'\s+', '', website)  # e.g. website anchor split over two <a> tags
    return phone, fax, email, website


In [ ]:
#------------------------------------------------ Begin_Main ----------------------------------------

for k, reg in enumerate(regdict):
    print(f"[INFO] : Working {k+1}/{len(regdict)} _({reg})_ {Typology[reg]}")
    rows_before = len(sqldict['Name'])

    resp = requests.get(regdict[reg], headers=headers, verify=False, timeout=60)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.content, 'lxml')

    main_ = soup.find('div', class_='region-content') or soup
    table_ = main_.find('table')  # first table = the institutions list

    for row in table_.find_all('tr'):
        td_cells = row.find_all('td')
        if len(td_cells) < 4:
            # header rows (3 cells) and sub-section rows like "(A) Licensed Commercial Banks" (2 cells)
            continue

        name_, address_ = extract_name_and_address(td_cells[1])
        if not name_ or name_.lower().startswith('name'):
            continue

        phone_, fax_, email_, website_ = extract_contacts(td_cells[2], td_cells[3])

        sqldict['Name'].append(name_)
        sqldict['Address_1'].append(address_)
        sqldict['Phone'].append(phone_)
        sqldict['Fax'].append(fax_)
        sqldict['Email'].append(email_)
        sqldict['Website'].append(website_)
        sqldict['ListLabel'].append(ListLabeldict[reg])
        sqldict['ListProcessDate'].append(processdate)
        sqldict['ListName'].append(Typology[reg])
        sqldict['RegCtry'].append(reg.split()[0])
        sqldict['RegCode'].append(reg.split()[1])
        sqldict['ListCode'].append(reg.split()[2])
        sqldict['RegulationType'].append('Regulated')

    sqldict = bourange_same_length_array(sqldict)
    rows_after = len(sqldict['Name'])
    print(f"[INFO] : {reg} collected {rows_after - rows_before} rows")
    sleep(1)

In [ ]:
#------------------------------------------------ Begin_writer and save df to excel  ----------------------------------------
os.chdir(scriptfolder)

df = pd.DataFrame(sqldict)

df = df[df['Name'] != '']

df.to_excel(os.path.join(scriptfolder, filename), sheet_name='SQL Ready', index=False)

print(f"[INFO] : Saved {len(df)} rows to {os.path.join(scriptfolder, filename)}")
print(df.groupby(['ListCode', 'ListName']).size())
